<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/R3_TS2_alignement_Superimpose_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# INSTALL
# ============================================================

!pip install -q morfeus-ml pandas numpy openpyxl


# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive', force_remount=True)


# ============================================================
# IMPORTS
# ============================================================

import os
import pandas as pd
import numpy as np

from morfeus import Sterimol, BuriedVolume, read_xyz


# ============================================================
# PATH
# ============================================================

xyz_folder = "/content/drive/MyDrive/xyz_files"


# ============================================================
# YOUR FIXED XYZ NUMBERING
#
# DDQ       = 1–15
# Substrate = 16–rest
# C2        = 18
# H on C2   = 43
# R3 first  = 44
# R3        = 44–last atom
# ============================================================

C2_ATOM = 18
R3_START_ATOM = 44


# ============================================================
# R3 NAMES
#
# If filename is not listed here, filename itself is used.
# ============================================================

R3_names = {

    "RR_Me.xyz": "Me",
    "RR_Et.xyz": "Et",
    "RR_iPr.xyz": "i-Pr",
    "RR_nBu.xyz": "n-Bu",
    "RR_tBu.xyz": "t-Bu",
    "RR_Cy.xyz": "Cy",
    "RR_CH2CO2Me.xyz": "CH2CO2Me",
    "RR_CMe2CO2Me.xyz": "CMe2CO2Me",
    "RR_allyl.xyz": "CH2CH=CH2",
    "RR_CMe2CHCH2.xyz": "CMe2CH=CH2",
    "RR_CH2_dioxolane.xyz": "CH2-1,3-dioxolane",
    "RR_CH2Ph.xyz": "CH2Ph",
    "RR_CH2CN.xyz": "CH2CN",
    "RR_Ph.xyz": "Phenyl",
    "RR_vinyl.xyz": "CH=CH2",
    "RR_alkyne.xyz": "C#CH"
}


# ============================================================
# STORAGE
# ============================================================

results = []


# ============================================================
# GET ALL XYZ FILES
# ============================================================

xyz_files = sorted([

    f for f in os.listdir(xyz_folder)

    if f.lower().endswith(".xyz")

])


print("Number of XYZ files:", len(xyz_files))


# ============================================================
# PROCESS EVERY XYZ
# ============================================================

for file in xyz_files:

    print("\n====================================================")
    print("Processing:", file)
    print("====================================================")

    path = os.path.join(xyz_folder, file)

    try:

        # ----------------------------------------------------
        # READ XYZ
        # ----------------------------------------------------

        elements, coordinates = read_xyz(path)

        n_atoms = len(elements)


        # ----------------------------------------------------
        # CHECK NUMBERING
        # ----------------------------------------------------

        if n_atoms < R3_START_ATOM:

            raise ValueError(
                f"Only {n_atoms} atoms found; "
                f"R3 should start at atom {R3_START_ATOM}."
            )


        # ----------------------------------------------------
        # DEFINE R3
        #
        # Because your numbering is standardized:
        #
        # R3 = 44 through the last atom
        # ----------------------------------------------------

        R3_atoms = list(
            range(R3_START_ATOM, n_atoms + 1)
        )


        print(
            "R3 atoms:",
            R3_atoms
        )


        # ----------------------------------------------------
        # CHECK C18-C44 DISTANCE
        # ----------------------------------------------------

        c18 = coordinates[C2_ATOM - 1]

        c44 = coordinates[R3_START_ATOM - 1]

        c18_c44_distance = np.linalg.norm(
            c18 - c44
        )


        print(
            f"C18-C44 distance = "
            f"{c18_c44_distance:.3f} Å"
        )


        # ====================================================
        # STERIMOL
        #
        # YOUR DEFINED STERIC AXIS:
        #
        # C18 --------------------> C44
        # C2                         first R3 carbon
        #
        # IMPORTANT:
        # Morfeus indices are 1-based.
        # ====================================================

        keep_atoms = set(
            [C2_ATOM] + R3_atoms
        )


        # Exclude DDQ + remaining substrate atoms
        excluded_atoms = [

            i

            for i in range(1, n_atoms + 1)

            if i not in keep_atoms

        ]


        sterimol = Sterimol(

            elements,
            coordinates,

            dummy_index=C2_ATOM,

            attached_index=R3_START_ATOM,

            excluded_atoms=excluded_atoms,

            radii_type="crc",

            n_rot_vectors=3600

        )


        # ----------------------------------------------------
        # STERIMOL VALUES
        # ----------------------------------------------------

        B1 = sterimol.B_1_value

        B5 = sterimol.B_5_value

        L = sterimol.L_value


        print(
            f"Sterimol B1 = {B1:.3f} Å"
        )

        print(
            f"Sterimol B5 = {B5:.3f} Å"
        )

        print(
            f"Sterimol L  = {L:.3f} Å"
        )


        # ====================================================
        # CUSTOM R3 %VBUR
        #
        # Center = C18
        #
        # Only R3 atoms are allowed to contribute.
        #
        # C18 itself is the center and is automatically
        # excluded from the buried-volume contribution.
        # ====================================================

        vbur_excluded_atoms = [

            i

            for i in range(1, n_atoms + 1)

            if i not in R3_atoms

        ]


        buried = BuriedVolume(

            elements,

            coordinates,

            metal_index=C2_ATOM,

            excluded_atoms=vbur_excluded_atoms,

            radius=3.5,

            include_hs=False,

            radii_type="bondi"

        )


        # ----------------------------------------------------
        # %VBUR
        # ----------------------------------------------------

        percent_vbur = buried.percent_buried_volume


        print(
            f"%Vbur = {percent_vbur:.3f} %"
        )


        # ====================================================
        # R3 NAME
        # ====================================================

        R3 = R3_names.get(

            file,

            os.path.splitext(file)[0]

        )


        # ====================================================
        # SAVE
        # ====================================================

        results.append({

            "File": file,

            "R3": R3,

            "C2 atom": C2_ATOM,

            "R3 first atom": R3_START_ATOM,

            "R3 atoms":
                f"{R3_START_ATOM}-{n_atoms}",

            "C18-C44 distance (Å)":
                c18_c44_distance,

            "B1 (Å)": B1,

            "B5 (Å)": B5,

            "L (Å)": L,

            "%Vbur": percent_vbur

        })


        print(
            "SUCCESS"
        )


    except Exception as e:

        print(
            f"ERROR in {file}: {e}"
        )


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(results)


# ============================================================
# SAVE CSV
# ============================================================

csv_file = os.path.join(

    xyz_folder,

    "R3_Sterimol_Vbur_results_new.csv"

)

df.to_csv(

    csv_file,

    index=False

)


# ============================================================
# SAVE EXCEL
# ============================================================

xlsx_file = os.path.join(

    xyz_folder,

    "R3_Sterimol_Vbur_results_new.xlsx"

)

df.to_excel(

    xlsx_file,

    index=False

)


# ============================================================
# DISPLAY
# ============================================================

print("\n")
print("====================================================")
print("CALCULATION COMPLETED")
print("====================================================")

print(
    "CSV   :", csv_file
)

print(
    "Excel :", xlsx_file
)

print("\nRESULTS:")

display(df)

Mounted at /content/drive
Number of XYZ files: 34

Processing: RR_CH2-Ph.xyz
R3 atoms: [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57]
C18-C44 distance = 1.529 Å
Sterimol B1 = 1.700 Å
Sterimol B5 = 5.853 Å
Sterimol L  = 6.870 Å


/tmp/ipykernel_784/375439761.py:289: DeprecationWarning: 'percent_buried_volume' is deprecated. Use 'fraction_buried_volume'.
  percent_vbur = buried.percent_buried_volume


%Vbur = 0.272 %
SUCCESS

Processing: RR_CH2-dioxane.xyz
R3 atoms: [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56]
C18-C44 distance = 1.529 Å
Sterimol B1 = 1.700 Å
Sterimol B5 = 5.246 Å
Sterimol L  = 6.142 Å
%Vbur = 0.264 %
SUCCESS

Processing: RR_CH2CN.xyz
R3 atoms: [44, 45, 46, 47, 48]
C18-C44 distance = 1.540 Å
Sterimol B1 = 1.700 Å
Sterimol B5 = 3.992 Å
Sterimol L  = 4.439 Å
%Vbur = 0.240 %
SUCCESS

Processing: RR_CH2CO2Me.xyz
R3 atoms: [44, 45, 46, 47, 48, 49, 50, 51, 52, 53]
C18-C44 distance = 1.530 Å
Sterimol B1 = 1.700 Å
Sterimol B5 = 4.957 Å
Sterimol L  = 5.778 Å
%Vbur = 0.265 %
SUCCESS

Processing: RR_CMe2CO2Me.xyz
R3 atoms: [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
C18-C44 distance = 1.553 Å
Sterimol B1 = 2.841 Å
Sterimol B5 = 5.068 Å
Sterimol L  = 5.513 Å
%Vbur = 0.370 %
SUCCESS

Processing: RR_Cy.xyz
R3 atoms: [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
C18-C44 distance = 1.539 Å
Sterimol B1 = 2.007 Å
Sterimol B5 = 3.676 

,File,R3,C2 atom,R3 first atom,R3 atoms,C18-C44 distance (Å),B1 (Å),B5 (Å),L (Å),%Vbur
0,RR_CH2-Ph.xyz,RR_CH2-Ph,18,44,44-57,1.529416,1.700000,5.852931,6.869516,0.271588
1,RR_CH2-dioxane.xyz,RR_CH2-dioxane,18,44,44-56,1.528873,1.700000,5.245980,6.141713,0.263849
2,RR_CH2CN.xyz,CH2CN,18,44,44-48,1.539665,1.700000,3.992223,4.438541,0.239675
3,RR_CH2CO2Me.xyz,CH2CO2Me,18,44,44-53,1.529564,1.700000,4.957261,5.777883,0.264792
4,RR_CMe2CO2Me.xyz,CMe2CO2Me,18,44,44-59,1.552641,2.841477,5.068311,5.512985,0.370091
5,RR_Cy.xyz,Cy,18,44,44-60,1.539015,2.006980,3.676088,6.749453,0.292554
6,RR_Et.xyz,Et,18,44,44-50,1.529597,1.700000,3.255715,4.676969,0.238871
7,RR_Me-Propane.xyz,RR_Me-Propane,18,44,44-56,1.537140,1.700000,4.502030,5.803826,0.268752
8,RR_Me.xyz,Me,18,44,44-47,1.525097,1.700000,2.129183,3.625097,0.183697
9,RR_Ph.xyz,Phenyl,18,44,44-54,1.519774,1.700000,3.272252,6.899519,0.277593
